# 02 — Eksperimen Klasik (Skenario 1–10)

SVM & Random Forest dengan feature engineering manual. Jalankan **berurutan** setelah `01_eda.ipynb`.

In [ ]:
# Google Colab Setup
# Jalankan sel ini PERTAMA di setiap session Colab baru.
import os, sys
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Sesuaikan jika nama folder di Google Drive berbeda
    PROJECT_PATH = '/content/drive/MyDrive/pcd-kelompok-17'
    os.chdir(PROJECT_PATH)
    sys.path.insert(0, PROJECT_PATH)
    print('Setup selesai. Project:', PROJECT_PATH)
else:
    print('Lokal - skip mount Drive.')


In [ ]:
# Install dependencies - jalankan sekali per session Colab
%pip install -q opencv-python-headless scipy scikit-image scikit-learn statsmodels matplotlib seaborn pandas tqdm joblib kagglehub tensorflow


In [ ]:
import os
import sys
from pathlib import Path

_in_colab = "COLAB_RELEASE_TAG" in os.environ
ROOT = Path.cwd() if _in_colab else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.experiments import (
    run_classical_scenario,
    run_mcnemar_pair,
    select_best_enhancement,
)
from src.utils import get_project_paths, load_splits, set_seed

set_seed(42)
paths = get_project_paths()
train, val, test = load_splits(paths["splits"])
cache_dir = paths["data_processed"]
metrics_dir = paths["metrics"]
figures_dir = paths["figures_confusion"]
models_dir = paths["models"]
print(len(train), len(val), len(test))


## Skenario 1–4: Perbandingan Enhancement

In [ ]:
val_f1_map = {}
scenario_results = {}

for sid in range(1, 5):
    print(f"\n=== Skenario {sid} ===")
    res = run_classical_scenario(
        sid, train, val, test,
        metrics_dir, figures_dir, models_dir, cache_dir,
    )
    val_f1_map[res["enhancement"]] = res["val_f1"]
    scenario_results[sid] = res
    print(f"Val F1: {res['val_f1']:.4f} | Test F1: {res['test_metrics']['f1_weighted']:.4f}")

best_enh = select_best_enhancement(val_f1_map, metrics_dir)
print(f"\nE* (enhancement terbaik): {best_enh}")


## Skenario 5–10: Segmentasi, Integrasi, Ablasi, RF

In [ ]:
for sid in range(5, 11):
    print(f"\n=== Skenario {sid} ===")
    res = run_classical_scenario(
        sid, train, val, test,
        metrics_dir, figures_dir, models_dir, cache_dir,
    )
    scenario_results[sid] = res
    print(f"Test F1: {res['test_metrics']['f1_weighted']:.4f}")


## Uji Signifikansi McNemar

In [ ]:
from src.utils import read_best_enhancement

best_enh = read_best_enhancement(metrics_dir)
best_sid = {"none": 1, "clahe": 2, "histeq": 3, "gamma": 4}[best_enh]
y_true = scenario_results[1]["y_test"]

run_mcnemar_pair(
    f"E*({best_enh}) vs S1",
    f"S{best_sid}", "S1",
    y_true,
    scenario_results[best_sid]["y_pred"],
    scenario_results[1]["y_pred"],
    metrics_dir,
)
run_mcnemar_pair(
    "S5 vs S1", "S5", "S1",
    y_true, scenario_results[5]["y_pred"], scenario_results[1]["y_pred"], metrics_dir,
)
run_mcnemar_pair(
    "S6 vs S1", "S6", "S1",
    y_true, scenario_results[6]["y_pred"], scenario_results[1]["y_pred"], metrics_dir,
)
print("McNemar (S11 vs S6) dijalankan di notebook 03.")


In [ ]:
import pandas as pd
sig_path = metrics_dir / "significance_tests.csv"
if sig_path.exists():
    display(pd.read_csv(sig_path))
